# Bayesian Prediction Band Export (Old Logic)

Export a Bayesian prediction-band pickle using the same posterior-draw and PDF-sampling logic as `plot_bayesian (2).ipynb`. This notebook keeps the local `plot_proper.ipynb` card/data setup, uses uniform PDF-replica sampling per posterior draw, and builds plain quantiles over the pooled predictive samples. The expensive Julia theory call is evaluated once per posterior draw; sampled PDF shifts are added afterward in NumPy.


In [1]:
from julia.api import Julia
from julia import Main

import os
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm


c:\Users\congyue zhang\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fit_name = "Final_old"
parameter_results_path = Path("bayesian parameters/bayesian_parameters.csv")
output_band_path = Path("prediction bands/bayesian_old.pkl")

band_alpha = 15.865
max_parameter_rows = None

data_selections = "by_experiment"  # "by_file" or "by_experiment"
selected_files = []
experiments = [
    "ATLAS_7",
    "ATLAS_8",
    "CDF_I",
    "CDF_II",
    "CMS_7",
    "CMS_8",
    "CMS_13",
    "D0_I",
    "D0_II",
    "D0_II_mu",
    "E288",
    "E605",
    "E772",
    "LHCb_7",
    "LHCb_8",
    "LHCb_13",
    "STAR",
]
file_excludes = [
    "E772/E772-5Q6.csv",
    "E772/E772-6Q7.csv",
    "E772/E772-7Q8.csv",
    "E772/E772-8Q9.csv",
]

approximate_total_xsec = True
data_uncertainty_only = False
use_pdf_shift = True

# Set this to an explicit parameter list if you want the exported `central` curve to use a
# hard-coded central-fit point instead of `Main.initial_params`.
central_params_phys_override = None


In [3]:
TMD_fitting_root = Path("../")

def include(name):
    path = (TMD_fitting_root / name).resolve()
    Main.eval(f'include(raw"{path.as_posix()}")')

include(f"Cards/{fit_name}.jl")
include(f"DY/DY_table_{Main.flavor_scheme}.jl")

file_root = Path(f"../Data/{Main.data_name}/Cutted/DY")
matrix_root = Path(f"../Data/{Main.data_name}/Covariance_Matrices/DY")
table_root = Path(f"../Tables/{Main.table_name}/DY")
total_root = Path(f"../Data/DY_total_xsec/{Main.pdf_name}.csv")
error_sets_root = Path(f"../Data/PDF_Matrices/{Main.error_sets_name}/DY")
pdf_diff_root = Path(f"../Data/PDF_Differences/{Main.error_sets_name}")

initial_params = np.asarray(Main.initial_params, dtype=float)

card_path = TMD_fitting_root / "Cards" / f"{fit_name}.jl"
card_text = card_path.read_text(encoding="utf-8")

struct_match = re.search(r"struct\s+Params_Struct(.*?)end", card_text, re.S)
if struct_match is None:
    raise ValueError(f"Could not find Params_Struct in {card_path}")

param_names = re.findall(
    r"([A-Za-z_][A-Za-z0-9_]*)\s*::\s*Float32",
    struct_match.group(1),
)
param_columns = [f"param_{i}" for i in range(len(param_names))]
param_info_df = pd.DataFrame(
    {
        "index": np.arange(len(param_names), dtype=int),
        "parameter": param_names,
        "initial_value": initial_params[: len(param_names)],
    }
)
display(param_info_df)


,index,parameter,initial_value
0,0,lambda1,0.023656
1,1,lambda2,1.054291
2,2,lambda3,-2.354365
3,3,logx0,-5.207703
4,4,sigx,1.103274
5,5,amp,-0.431106
6,6,BNP,1.494665
7,7,c0,0.070013
8,8,c1,0.027637


In [4]:
def to_float64(df):
    num_cols = df.select_dtypes(include=["number"]).columns
    if len(num_cols):
        df[num_cols] = df[num_cols].astype("float64")
    return df

if data_selections == "by_file":
    file_names = list(selected_files)
else:
    file_names = []
    for experiment in experiments:
        exp_dir = file_root / experiment
        for path in sorted(exp_dir.glob("*.csv")):
            rel = f"{experiment}/{path.name}"
            if rel in file_excludes:
                continue
            file_names.append(rel)

print(f"Loaded {len(file_names)} datasets")
display(pd.Series(file_names, name="file"))

data_list = {}
matrix_data_list = {}
matrix_total_list = {}
matrix_inv_list = {}
data_vectors = {}
data_errors = {}

df_total_xsec = to_float64(pd.read_csv(total_root))
total_xsec_lookup = dict(zip(df_total_xsec["name"], df_total_xsec["total_xsec"]))

for file in tqdm(file_names, desc="Reading data"):
    data_df = to_float64(pd.read_csv(file_root / file))
    matrix_data = to_float64(pd.read_csv(matrix_root / file))
    matrix_data_list[file] = matrix_data

    matrix_pdf_path = error_sets_root / file
    if data_uncertainty_only or (not matrix_pdf_path.exists()):
        matrix_total = matrix_data.copy()
    else:
        matrix_total = matrix_data + to_float64(pd.read_csv(matrix_pdf_path))

    short_name = Path(file).stem
    if short_name in total_xsec_lookup:
        data_df["total_xsec"] = float(total_xsec_lookup[short_name])

    data_list[file] = data_df
    matrix_total_list[file] = matrix_total
    matrix_inv_list[file] = np.linalg.inv(np.asarray(matrix_total, dtype=float))
    data_vectors[file] = data_df["xsec"].to_numpy(dtype=float)
    data_errors[file] = np.sqrt(np.clip(np.diag(np.asarray(matrix_data, dtype=float)), 0.0, None))

file_lengths = {file: len(data_list[file]) for file in file_names}

def get_chi2dN(df_predictions):
    chi2_total = 0.0
    n_total = 0
    for file in file_names:
        diff_xsec = data_vectors[file] - np.asarray(df_predictions[file], dtype=float)
        chi2_total += float(diff_xsec @ matrix_inv_list[file] @ diff_xsec)
        n_total += len(diff_xsec)
    return chi2_total / n_total if n_total else np.nan


Loaded 57 datasets


0       ATLAS_7/ATLAS7-00y10.csv
1       ATLAS_7/ATLAS7-10y20.csv
2       ATLAS_7/ATLAS7-20y24.csv
3       ATLAS_8/ATLAS8-00y04.csv
4       ATLAS_8/ATLAS8-04y08.csv
5       ATLAS_8/ATLAS8-08y12.csv
6     ATLAS_8/ATLAS8-116Q150.csv
7       ATLAS_8/ATLAS8-12y16.csv
8       ATLAS_8/ATLAS8-16y20.csv
9       ATLAS_8/ATLAS8-20y24.csv
10      ATLAS_8/ATLAS8-46Q66.csv
11                CDF_I/CDF1.csv
12               CDF_II/CDF2.csv
13                CMS_7/CMS7.csv
14                CMS_8/CMS8.csv
15        CMS_13/CMS13-00y04.csv
16        CMS_13/CMS13-04y08.csv
17        CMS_13/CMS13-08y12.csv
18      CMS_13/CMS13-106Q170.csv
19        CMS_13/CMS13-12y16.csv
20        CMS_13/CMS13-16y24.csv
21      CMS_13/CMS13-170Q350.csv
22     CMS_13/CMS13-350Q1000.csv
23                  D0_I/D01.csv
24                 D0_II/D02.csv
25            D0_II_mu/D02mu.csv
26         E288/E228-200-4Q5.csv
27         E288/E228-200-5Q6.csv
28         E288/E228-200-6Q7.csv
29         E288/E228-200-7Q8.csv
30        

Reading data: 100%|██████████| 57/57 [00:01<00:00, 39.03it/s]


In [5]:
def _norm(path):
    return os.path.normpath(str(path)).replace("\\", "/")

def prediction_reformat(predictions):
    preds = {_norm(k): v for k, v in predictions.items()}
    df_predictions = {}

    for file in file_names:
        n_points = file_lengths[file]
        base = os.path.splitext(file)[0]
        xs = []

        for i in range(n_points):
            candidates = [
                _norm(table_root / f"{base}/{i}.jls"),
                _norm(table_root / f"{base}/{i}.jld2"),
            ]
            for candidate in candidates:
                if candidate in preds:
                    xs.append(preds[candidate])
                    break
            else:
                raise KeyError(f"Missing prediction entry for {file} point {i}")

        xs = np.asarray(xs, dtype=float)
        if approximate_total_xsec and Path(file).stem in total_xsec_lookup and xs.sum() != 0.0:
            data_xsec = data_vectors[file]
            xs = xs * (data_xsec.sum() / xs.sum())

        df_predictions[file] = xs

    return df_predictions

def _pdf_dataset_key(file):
    return str(Path(file).with_suffix("")).replace("\\", "/")

def load_pdf_shift_replica(path):
    with path.open("rb") as handle:
        pdf_diff_dict = pickle.load(handle)

    pdf_shift_list = {}
    for file in file_names:
        prefix = f"{_pdf_dataset_key(file)}/"
        values = []
        for i in range(file_lengths[file]):
            key = f"{prefix}{i}"
            if key not in pdf_diff_dict:
                raise KeyError(f"Missing PDF-difference entry: {key}")
            values.append(pdf_diff_dict[key])
        pdf_shift_list[file] = np.asarray(values, dtype=float)

    return pdf_shift_list

def set_params_and_predict(full_params):
    params_cl = Main.Params_Struct(*[np.float32(x) for x in full_params])
    Main.set_params(Main.VRAM, params_cl)
    predictions, eval_time = Main.xsec_dict(Main.rel_paths, Main.VRAM)
    return prediction_reformat(predictions), float(eval_time)

if use_pdf_shift and pdf_diff_root.exists():
    pdf_diff_paths = sorted(pdf_diff_root.glob("*.pkl"), key=lambda path: int(path.stem))
else:
    pdf_diff_paths = []

if use_pdf_shift and pdf_diff_paths:
    pdf_shift_replicas = {
        int(path.stem): load_pdf_shift_replica(path)
        for path in tqdm(pdf_diff_paths, desc="Loading PDF shifts")
    }
    pdf_replica_ids = sorted(pdf_shift_replicas)
    pdf_shift_stack = {
        file: np.stack([pdf_shift_replicas[idx][file] for idx in pdf_replica_ids], axis=0)
        for file in file_names
    }
else:
    pdf_replica_ids = [0]
    pdf_shift_stack = {
        file: np.zeros((1, file_lengths[file]), dtype=float)
        for file in file_names
    }

n_pdf_replicas = len(pdf_replica_ids)
print(f"Using {n_pdf_replicas} PDF replicas with equal weight per (parameter row, PDF replica) pair")


Loading PDF shifts: 100%|██████████| 100/100 [00:01<00:00, 92.35it/s]

Using 100 PDF replicas with equal weight per (parameter row, PDF replica) pair


In [6]:
def load_parameter_results(path):
    df = pd.read_csv(path)

    named_schema = all(name in df.columns for name in param_names)
    indexed_schema = all(col in df.columns for col in param_columns)

    if named_schema:
        for name in param_names:
            df[name] = pd.to_numeric(df[name], errors="coerce")
    elif indexed_schema:
        for col in param_columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        for col, name in zip(param_columns, param_names):
            df[name] = df[col]
    else:
        raise ValueError(
            f"{path} does not contain either named parameter columns {param_names} "
            f"or indexed columns {param_columns}"
        )

    if "source_index" in df.columns:
        df["source_index"] = pd.to_numeric(df["source_index"], errors="coerce")
        df = df.sort_values("source_index")

    df = df.dropna(subset=param_names).reset_index(drop=True)

    if max_parameter_rows is not None and len(df) > max_parameter_rows:
        df = df.head(max_parameter_rows).copy()

    return df

parameter_results_df = load_parameter_results(parameter_results_path)
print(f"Loaded {len(parameter_results_df)} parameter rows from {parameter_results_path}")
display_cols = [col for col in ["source_index", "log_prob"] if col in parameter_results_df.columns]
display(parameter_results_df[display_cols + param_names].head())

if central_params_phys_override is None:
    theta_central_full_phys = initial_params.copy()
else:
    theta_central_full_phys = np.asarray(central_params_phys_override, dtype=float)
    if len(theta_central_full_phys) != len(initial_params):
        raise ValueError(
            f"central_params_phys_override must have length {len(initial_params)}, got {len(theta_central_full_phys)}"
        )

central_predictions, central_eval_time = set_params_and_predict(theta_central_full_phys)
print(f"One central evaluation took {1000.0 * central_eval_time:.1f} ms")

n_parameter_rows = len(parameter_results_df)
central_prediction_bank = {
    file: np.empty((n_parameter_rows, file_lengths[file]), dtype=float)
    for file in file_names
}
eval_times_ms = []

for row_idx, row in enumerate(
    tqdm(parameter_results_df[param_names].itertuples(index=False, name=None), total=n_parameter_rows, desc="Evaluating parameter draws")
):
    full_params = np.asarray(row, dtype=float)
    df_predictions, eval_time = set_params_and_predict(full_params)
    eval_times_ms.append(1000.0 * eval_time)
    for file in file_names:
        central_prediction_bank[file][row_idx] = np.asarray(df_predictions[file], dtype=float)

eval_summary_df = pd.DataFrame({"eval_time_ms": eval_times_ms})
display(eval_summary_df.describe())


Loaded 1000 parameter rows from bayesian parameters\bayesian_parameters.csv


,source_index,log_prob,lambda1,lambda2,lambda3,logx0,sigx,amp,BNP,c0,c1
0,9,-186.607928,0.004001,1.066987,-2.639572,-5.354135,1.032392,-0.464033,1.915794,0.059724,0.022447
1,41,-184.315792,0.002548,1.122492,-2.657517,-5.301107,1.660783,-0.429345,1.741697,0.064500,0.026654
2,51,-185.509553,-0.008229,1.084817,-2.818570,-5.745763,1.447606,-0.570028,1.801397,0.065770,0.027093
3,76,-188.391945,0.072423,1.479139,-3.174244,-5.034232,1.974748,-0.435425,1.671421,0.067712,0.027458
4,85,-184.582612,0.000150,1.076239,-2.828981,-5.411874,1.594033,-0.441847,1.937314,0.064434,0.022292


One central evaluation took 294.0 ms


Evaluating parameter draws: 100%|██████████| 1000/1000 [01:23<00:00, 11.95it/s]


,eval_time_ms
count,1000.000000
mean,39.178628
std,18.786132
min,37.092900
25%,37.429325
50%,37.593000
75%,37.883325
max,380.466300


In [7]:
ensemble_summary_df = pd.DataFrame(
    {
        "n_parameter_rows": [n_parameter_rows],
        "n_pdf_replicas": [n_pdf_replicas],
        "n_total_pairs": [int(n_parameter_rows * n_pdf_replicas)],
    }
)
display(ensemble_summary_df)


,n_parameter_rows,n_pdf_replicas,n_total_pairs
0,1000,100,100000


In [8]:
quantiles = np.array([band_alpha / 100.0, 0.5, 1.0 - band_alpha / 100.0], dtype=float)

band_payload = {}

for file in tqdm(file_names, desc="Summarizing prediction bands"):
    central_array = central_prediction_bank[file]
    shift_array = pdf_shift_stack[file]
    n_points = file_lengths[file]

    pooled_array = (
        central_array[:, None, :] + shift_array[None, :, :]
    ).reshape(-1, n_points)

    pred_low, pred_median, pred_up = np.quantile(pooled_array, quantiles, axis=0)
    pred_mean = pooled_array.mean(axis=0)

    data_vals = data_vectors[file]
    data_err = data_errors[file]
    qT_vals = data_list[file]["qT_mean"].to_numpy(dtype=float)
    central_vals = np.asarray(central_predictions[file], dtype=float)

    ratio_low = np.divide(pred_low, data_vals, out=np.full(n_points, np.nan), where=data_vals != 0.0)
    ratio_median = np.divide(pred_median, data_vals, out=np.full(n_points, np.nan), where=data_vals != 0.0)
    ratio_up = np.divide(pred_up, data_vals, out=np.full(n_points, np.nan), where=data_vals != 0.0)
    ratio_mean = np.divide(pred_mean, data_vals, out=np.full(n_points, np.nan), where=data_vals != 0.0)
    central_ratio = np.divide(central_vals, data_vals, out=np.full(n_points, np.nan), where=data_vals != 0.0)
    ratio_err = np.divide(data_err, np.abs(data_vals), out=np.full(n_points, np.nan), where=data_vals != 0.0)

    band_payload[file] = pd.DataFrame(
        {
            "qT": qT_vals,
            "data": data_vals,
            "data_err": data_err,
            "central": central_vals,
            "pred_low": pred_low,
            "pred_up": pred_up,
            "pred_mean": pred_mean,
            "pred_median": pred_median,
            "pred_lo": pred_low,
            "pred_hi": pred_up,
            "pred_mid": pred_median,
            "ratio_low": ratio_low,
            "ratio_up": ratio_up,
            "ratio_mean": ratio_mean,
            "ratio_median": ratio_median,
            "ratio_lo": ratio_low,
            "ratio_hi": ratio_up,
            "ratio_mid": ratio_median,
            "central_ratio": central_ratio,
            "ratio_err": ratio_err,
        }
    )

output_band_path.parent.mkdir(parents=True, exist_ok=True)
with output_band_path.open("wb") as handle:
    pickle.dump(band_payload, handle, protocol=pickle.HIGHEST_PROTOCOL)

summary_df = pd.DataFrame(
    {
        "n_parameter_rows": [n_parameter_rows],
        "n_pdf_replicas": [n_pdf_replicas],
        "n_total_pairs": [int(n_parameter_rows * n_pdf_replicas)],
        "band_alpha_percent": [100.0 - 2.0 * band_alpha],
        "output_path": [str(output_band_path)],
    }
)
display(summary_df)

preview_file = file_names[0]
print(f"Preview dataset: {preview_file}")
display(band_payload[preview_file].head())


Summarizing prediction bands: 100%|██████████| 57/57 [00:01<00:00, 56.81it/s]


,n_parameter_rows,n_pdf_replicas,n_total_pairs,band_alpha_percent,output_path
0,1000,100,100000,68.27,prediction bands\bayesian_old.pkl


Preview dataset: ATLAS_7/ATLAS7-00y10.csv


,qT,data,data_err,central,pred_low,pred_up,pred_mean,pred_median,pred_lo,pred_hi,pred_mid,ratio_low,ratio_up,ratio_mean,ratio_median,ratio_lo,ratio_hi,ratio_mid,central_ratio,ratio_err
0,1.0,0.02861,0.000177,0.028178,0.027691,0.028478,0.028072,0.028082,0.027691,0.028478,0.028082,0.967875,0.995390,0.981180,0.981537,0.967875,0.995390,0.981537,0.984885,0.006179
1,3.0,0.05874,0.000302,0.057968,0.057583,0.058384,0.057953,0.057966,0.057583,0.058384,0.057966,0.980303,0.993941,0.986596,0.986827,0.980303,0.993941,0.986827,0.986860,0.005145
2,5.0,0.05834,0.000278,0.058275,0.058179,0.058330,0.058256,0.058254,0.058179,0.058330,0.058254,0.997241,0.999837,0.998552,0.998519,0.997241,0.999837,0.998519,0.998893,0.004773
3,7.0,0.04972,0.000239,0.050057,0.049841,0.050267,0.050062,0.050057,0.049841,0.050267,0.050057,1.002432,1.011003,1.006884,1.006770,1.002432,1.011003,1.006770,1.006769,0.004805
4,9.0,0.04106,0.000206,0.041518,0.041303,0.041820,0.041581,0.041577,0.041303,0.041820,0.041577,1.005925,1.018510,1.012687,1.012600,1.005925,1.018510,1.012600,1.011142,0.005028
